<!-- RAG with PDF data extraction  -->

In [3]:
!pip install pypdf

In [4]:
import os

from dotenv import load_dotenv
load = load_dotenv(".env")

In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2202
)


In [17]:
# Ectracting PDF files
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "attention.pdf"
pdf2 = "LLMForgetting.pdf"
pdf3 = "TestingAndEvaluatingLLM.pdf"
pdf4 = "user_Profile.pdf.pdf"

pdfFiles = [pdf1, pdf2, pdf3, pdf4]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(f"Total number of pages in all PDFs: {len(documents)}")

Total number of pages in all PDFs: 262


In [19]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


663

In [21]:
# Embedding the chunks

from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings

# Ollama can fail when a large document list is sent in one request.
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


class BatchedOllamaEmbeddings(Embeddings):
    def __init__(self, embedding_model, batch_size=8):
        self.embedding_model = embedding_model
        self.batch_size = batch_size

    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start:start + self.batch_size]
            vectors.extend(self.embedding_model.embed_documents(batch))
        return vectors

    def embed_query(self, text):
        return self.embedding_model.embed_query(text)


embeddings = BatchedOllamaEmbeddings(ollama_embeddings)

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

768
768


In [22]:
# Vector store

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_v2",
)

In [32]:
# Retrieve relevant chunks

from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_v2",
    embedding_function=embeddings,
)

question = "What is my overall AI career direction?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

score = vector_store.similarity_search_with_score("What is my current role?")


score[0]

# for doc in retrieved_docs:
#     print(doc)

(Document(id='0fe521a6-a107-42c1-b52e-9591581873b8', metadata={'trapped': '/False', 'producer': 'ReportLab PDF Library - (opensource)', 'page': 0, 'title': 'User Knowledge Base for RAG', 'keywords': '', 'creationdate': '2026-09-07T11:56:22+00:00', 'source': 'user_Profile.pdf.pdf', 'creator': '(unspecified)', 'author': 'Generated from conversation context', 'page_label': '1', 'total_pages': 9, 'subject': '(unspecified)', 'moddate': '2026-09-07T11:56:22+00:00', 'start_index': 0}, page_content="User Knowledge Base for RAG\nPurpose: A structured knowledge document containing the non-sensitive information available from the user's profile, prior\nlearning context, projects, preferences, and goals. It is intended to be loaded into a vector database and retrieved through a\nRAG pipeline.\nImportant: This document is a synthesized knowledge base, not a verbatim transcript. It deliberately excludes sensitive\npersonal data such as health conditions, political affiliation, religion, precise loca

In [29]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the documents.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

The user's overall AI career direction focuses on integrating AI systems with practical software-testing workflows. This includes specializing in testing and evaluating AI systems (e.g., retrieval quality, hallucination detection, security testing) while combining tools like Playwright, Appium, CI/CD, vector databases (ChromaDB), RAG systems, and local models (Ollama). They aim to strengthen JavaScript/TypeScript skills and develop LLM applications within testing frameworks, emphasizing real-world application over theoretical study. Key technologies include MCP servers/clients, agent workflows, and CI/CD pipelines for distributed testing.


In [ ]:
# Retrivers in langchain


retriver = vector_store.as_retriever(
    search_Type = "similarity"
)